# Backtest Results — Sigmoid Confidence Formula

Replays the trading formula against collected data, analyzes PnL,
and explores parameter sensitivity.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from itertools import product

from analysis.data_loader import load_all_intervals, load_all_snapshots
from analysis.backtest import (
    run_backtest, results_to_dataframe,
    FormulaParams, GateConfig,
)

plt.rcParams.update({
    'figure.figsize': (12, 5),
    'axes.grid': True,
    'grid.alpha': 0.3,
})

TF = '5m'

In [ ]:
intervals = load_all_intervals(TF)
snapshots = load_all_snapshots(TF)
print(f'Loaded {len(intervals)} intervals, {len(snapshots)} snapshot rows')

## 1. Default Parameters

In [ ]:
bt = run_backtest(snapshots, intervals, timeframe=TF)
summary = bt.summary()

print('=== Default params (a=5, b=3, offset=4, F=0.6) ===')
for k, v in summary.items():
    if isinstance(v, float):
        print(f'  {k}: {v:.4f}')
    else:
        print(f'  {k}: {v}')

df = results_to_dataframe(bt)
entered = df[df['entered']].copy()

In [ ]:
# Per-asset breakdown
print('=== Per-asset breakdown ===')
asset_stats = entered.groupby('asset').agg(
    entries=('pnl', 'count'),
    wins=('win', 'sum'),
    win_rate=('win', 'mean'),
    total_pnl=('pnl', 'sum'),
    avg_pnl=('pnl', 'mean'),
    avg_entry_conf=('entry_confidence', 'mean'),
    avg_entry_price=('entry_price', 'mean'),
    flips=('flipped', 'sum'),
).round(4)
print(asset_stats.to_string())

## 2. PnL Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Cumulative PnL
entered_sorted = entered.sort_values('interval_id')
for asset in sorted(entered['asset'].unique()):
    sub = entered_sorted[entered_sorted['asset'] == asset]
    axes[0].plot(range(len(sub)), sub['pnl'].cumsum(), label=asset.upper())
axes[0].set_xlabel('Trade #')
axes[0].set_ylabel('Cumulative PnL ($)')
axes[0].set_title('Cumulative PnL by asset')
axes[0].legend()
axes[0].axhline(0, color='black', linewidth=0.5)

# PnL distribution
axes[1].hist(entered['pnl'], bins=30, alpha=0.7, edgecolor='black')
axes[1].set_xlabel('PnL per interval ($)')
axes[1].set_ylabel('Count')
axes[1].set_title(f'PnL distribution (mean={entered["pnl"].mean():.3f})')
axes[1].axvline(0, color='red', linestyle='--')

# Win rate by entry confidence
entered['conf_bin'] = pd.cut(entered['entry_confidence'], bins=8)
cal = entered.groupby('conf_bin', observed=True)['win'].agg(['mean', 'count'])
cal = cal[cal['count'] >= 3]
bin_centers = [interval.mid for interval in cal.index]
axes[2].bar(range(len(cal)), cal['mean'], color='steelblue')
axes[2].set_xticks(range(len(cal)))
axes[2].set_xticklabels([f'{x:.2f}' for x in bin_centers], rotation=45)
axes[2].set_xlabel('Entry confidence')
axes[2].set_ylabel('Win rate')
axes[2].set_title('Win rate by confidence at entry')
axes[2].axhline(0.5, color='red', linestyle='--', alpha=0.5)
for i, row in enumerate(cal.itertuples()):
    axes[2].text(i, row.mean + 0.02, f'n={row.count}', ha='center', fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# Analyze losses: why did we lose?
losses = entered[entered['win'] == False].copy()
wins = entered[entered['win'] == True].copy()

print('=== Win vs Loss comparison ===')
for label, sub in [('Wins', wins), ('Losses', losses)]:
    print(f'\n{label} (n={len(sub)}):')
    print(f'  Avg entry second:     {sub["entry_sec"].mean():.0f}')
    print(f'  Avg entry confidence:  {sub["entry_confidence"].mean():.3f}')
    print(f'  Avg entry price:       ${sub["entry_price"].mean():.3f}')
    print(f'  Avg |entry delta|:     {sub["entry_delta"].abs().mean():.6f}')
    print(f'  Flipped:               {sub["flipped"].sum()}/{len(sub)}')

## 3. Entry Timing Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Entry second distribution by outcome
for outcome, color in [(True, 'green'), (False, 'red')]:
    sub = entered[entered['win'] == outcome]
    label = 'Win' if outcome else 'Loss'
    axes[0].hist(sub['entry_sec'], bins=20, alpha=0.6, label=f'{label} (n={len(sub)})', color=color)
axes[0].set_xlabel('Entry second')
axes[0].set_ylabel('Count')
axes[0].set_title('Entry timing by outcome')
axes[0].legend()

# Entry price vs outcome
colors = entered['win'].map({True: 'green', False: 'red'})
axes[1].scatter(entered['entry_price'], entered['pnl'], c=colors, alpha=0.5, s=30)
axes[1].set_xlabel('Entry token price ($)')
axes[1].set_ylabel('PnL ($)')
axes[1].set_title('Entry price vs PnL')
axes[1].axhline(0, color='black', linewidth=0.5)

plt.tight_layout()
plt.show()

## 4. Parameter Sensitivity (Grid Search)

Test a range of parameter values to find which direction improves results.
This is a coarse grid — not a full optimization.

In [ ]:
# Grid search over key parameters
a_values = [3, 5, 8, 12]
b_values = [1, 3, 5]
offset_values = [2, 4, 6]

results = []
total = len(a_values) * len(b_values) * len(offset_values)
i = 0

for a_val, b_val, off_val in product(a_values, b_values, offset_values):
    i += 1
    params = FormulaParams(a=a_val, b=b_val, offset=off_val)
    bt = run_backtest(snapshots, intervals, params=params, timeframe=TF)
    s = bt.summary()
    s['a'] = a_val
    s['b'] = b_val
    s['offset'] = off_val
    results.append(s)

grid_df = pd.DataFrame(results)
print(f'Tested {len(grid_df)} parameter combinations')

In [ ]:
# Show top 10 by PnL and by win rate (min 20 entries)
viable = grid_df[grid_df['entered'] >= 20].copy()

print('=== Top 10 by total PnL (min 20 entries) ===')
top_pnl = viable.nlargest(10, 'total_pnl')
print(top_pnl[['a', 'b', 'offset', 'entered', 'wins', 'win_rate', 'total_pnl', 'profit_factor']].to_string(index=False))

print('\n=== Top 10 by win rate (min 20 entries) ===')
top_wr = viable.nlargest(10, 'win_rate')
print(top_wr[['a', 'b', 'offset', 'entered', 'wins', 'win_rate', 'total_pnl', 'profit_factor']].to_string(index=False))

print('\n=== Bottom 5 (worst PnL) ===')
bottom = viable.nsmallest(5, 'total_pnl')
print(bottom[['a', 'b', 'offset', 'entered', 'wins', 'win_rate', 'total_pnl', 'profit_factor']].to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# PnL vs a (averaged over other params)
for b_val in b_values:
    sub = viable[viable['b'] == b_val]
    grouped = sub.groupby('a')['total_pnl'].mean()
    axes[0].plot(grouped.index, grouped.values, 'o-', label=f'b={b_val}')
axes[0].set_xlabel('a (delta sensitivity)')
axes[0].set_ylabel('Avg total PnL ($)')
axes[0].set_title('PnL vs a')
axes[0].legend()
axes[0].axhline(0, color='black', linewidth=0.5)

# PnL vs b
for a_val in a_values:
    sub = viable[viable['a'] == a_val]
    grouped = sub.groupby('b')['total_pnl'].mean()
    axes[1].plot(grouped.index, grouped.values, 'o-', label=f'a={a_val}')
axes[1].set_xlabel('b (time weight)')
axes[1].set_ylabel('Avg total PnL ($)')
axes[1].set_title('PnL vs b')
axes[1].legend()
axes[1].axhline(0, color='black', linewidth=0.5)

# PnL vs offset
for a_val in a_values:
    sub = viable[viable['a'] == a_val]
    grouped = sub.groupby('offset')['total_pnl'].mean()
    axes[2].plot(grouped.index, grouped.values, 'o-', label=f'a={a_val}')
axes[2].set_xlabel('offset (sigmoid centering)')
axes[2].set_ylabel('Avg total PnL ($)')
axes[2].set_title('PnL vs offset')
axes[2].legend()
axes[2].axhline(0, color='black', linewidth=0.5)

plt.tight_layout()
plt.show()

In [ ]:
# Entries vs win rate tradeoff
fig, ax = plt.subplots(figsize=(10, 6))

scatter = ax.scatter(
    viable['entered'], viable['win_rate'],
    c=viable['total_pnl'], cmap='RdYlGn', s=60, alpha=0.8,
    edgecolors='black', linewidth=0.5
)
plt.colorbar(scatter, label='Total PnL ($)')
ax.set_xlabel('Number of entries')
ax.set_ylabel('Win rate')
ax.set_title('Entries vs Win Rate (color = PnL)')
ax.axhline(0.5, color='red', linestyle='--', alpha=0.5)

# Annotate best PnL point
best = viable.loc[viable['total_pnl'].idxmax()]
ax.annotate(
    f'a={best["a"]:.0f} b={best["b"]:.0f} off={best["offset"]:.0f}',
    (best['entered'], best['win_rate']),
    textcoords='offset points', xytext=(10, 10),
    arrowprops=dict(arrowstyle='->', color='black'),
    fontsize=9, fontweight='bold',
)

plt.tight_layout()
plt.show()

## 5. Best-Params Deep Dive

In [ ]:
# Re-run with best params and analyze
best_row = viable.loc[viable['total_pnl'].idxmax()]
best_params = FormulaParams(
    a=best_row['a'],
    b=best_row['b'],
    offset=best_row['offset'],
)
print(f'Best params: a={best_params.a}, b={best_params.b}, offset={best_params.offset}')

bt_best = run_backtest(snapshots, intervals, params=best_params, timeframe=TF)
df_best = results_to_dataframe(bt_best)
entered_best = df_best[df_best['entered']].copy()

print(f'\n=== Best params results ===')
for k, v in bt_best.summary().items():
    if isinstance(v, float):
        print(f'  {k}: {v:.4f}')
    else:
        print(f'  {k}: {v}')

print(f'\n=== Per-asset ===')
print(entered_best.groupby('asset').agg(
    entries=('pnl', 'count'),
    win_rate=('win', 'mean'),
    total_pnl=('pnl', 'sum'),
).round(4).to_string())

In [ ]:
# Compare default vs best cumulative PnL
bt_default = run_backtest(snapshots, intervals, timeframe=TF)
df_default = results_to_dataframe(bt_default)

fig, ax = plt.subplots(figsize=(12, 5))

default_entered = df_default[df_default['entered']].sort_values('interval_id')
best_entered = df_best[df_best['entered']].sort_values('interval_id')

ax.plot(range(len(default_entered)), default_entered['pnl'].cumsum(),
        label=f'Default (a=5,b=3,off=4) — ${default_entered["pnl"].sum():.2f}', alpha=0.8)
ax.plot(range(len(best_entered)), best_entered['pnl'].cumsum(),
        label=f'Best (a={best_params.a},b={best_params.b},off={best_params.offset}) — ${best_entered["pnl"].sum():.2f}', alpha=0.8)
ax.set_xlabel('Trade #')
ax.set_ylabel('Cumulative PnL ($)')
ax.set_title('Default vs Best Parameters')
ax.legend()
ax.axhline(0, color='black', linewidth=0.5)

plt.tight_layout()
plt.show()

## 6. Gate Sensitivity

In [ ]:
# Test different gate configurations with best params
gate_configs = [
    ('Default ($0.65-$0.85)', GateConfig()),
    ('Wider ($0.55-$0.90)', GateConfig(min_token_price=0.55, max_token_price=0.90)),
    ('Tighter ($0.70-$0.80)', GateConfig(min_token_price=0.70, max_token_price=0.80)),
    ('High only ($0.75-$0.90)', GateConfig(min_token_price=0.75, max_token_price=0.90)),
    ('No depth gate', GateConfig(min_depth=0.0)),
    ('High depth ($50+)', GateConfig(min_depth=50.0)),
]

gate_results = []
for name, gates in gate_configs:
    bt = run_backtest(snapshots, intervals, params=best_params, gates=gates, timeframe=TF)
    s = bt.summary()
    s['config'] = name
    gate_results.append(s)

gate_df = pd.DataFrame(gate_results)
print(gate_df[['config', 'entered', 'wins', 'win_rate', 'total_pnl', 'profit_factor']].to_string(index=False))

## 7. Key Takeaways

Summary observations from this initial backtest. These are preliminary —
based on a single day of heavily up-biased data (~88% up resolutions).

In [ ]:
print('=== Data limitations ===')
print(f'- Only 1 day of data ({len(intervals)} intervals)')
print(f'- Heavy up bias: {(intervals["resolution"] == "up").sum()}/{len(intervals[intervals["resolution"].isin(["up","down"])])} resolved up')
print(f'- 88.8% cross-asset agreement means ~1 independent bet, not 4')
print(f'- Need 2000+ intervals for reliable calibration')

print(f'\n=== What the default params show ===')
default_s = bt_default.summary()
print(f'- Win rate {default_s["win_rate"]:.1%} with avg confidence {default_s["avg_confidence_at_entry"]:.1%}')
print(f'- PnL: ${default_s["total_pnl"]:.2f} across {default_s["entered"]} entries')
print(f'- Fees are {default_s["total_fees"]:.2f}$ total — small relative to PnL')
print(f'- The formula enters too aggressively — confidence is high but outcomes dont match')

print(f'\n=== Next steps ===')
print('- Collect more days of data (especially ranging/down days)')
print('- Fine-tune grid search around promising parameter regions')
print('- Consider separate params per asset or per vol regime')
print('- Investigate if early delta direction is inherently unreliable')
print('- Test maker vs taker entry strategies')